1. Kütüphane Yükleme ve Veri Seti Okuma
Bu blokta, veri manipülasyonu, makine öğrenimi modelleme (özellik seçimi, pipeline, sınıflandırma modelleri) ve performans değerlendirmesi için gerekli tüm kütüphaneler yüklenir.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [2]:
df = pd.read_csv("DataSet/googleplaystore.csv")
print(df.shape)
print(df.head())

(10841, 13)
                                                 App        Category  Rating  \
0     Photo Editor & Candy Camera & Grid & ScrapBook  ART_AND_DESIGN     4.1   
1                                Coloring book moana  ART_AND_DESIGN     3.9   
2  U Launcher Lite – FREE Live Cool Themes, Hide ...  ART_AND_DESIGN     4.7   
3                              Sketch - Draw & Paint  ART_AND_DESIGN     4.5   
4              Pixel Draw - Number Art Coloring Book  ART_AND_DESIGN     4.3   

  Reviews  Size     Installs  Type Price Content Rating  \
0     159   19M      10,000+  Free     0       Everyone   
1     967   14M     500,000+  Free     0       Everyone   
2   87510  8.7M   5,000,000+  Free     0       Everyone   
3  215644   25M  50,000,000+  Free     0           Teen   
4     967  2.8M     100,000+  Free     0       Everyone   

                      Genres Last Updated         Current Ver   Android Ver  
0               Art & Design     7-Jan-18               1.0.0  4.0.3 and u

2. Veri Temizleme Fonksiyonları ve Özellik Mühendisliği
Bu bloklarda, metinsel sütunlardaki birimleri sayısal değerlere dönüştürmek ve hedef değişkeni oluşturmak için yardımcı fonksiyonlar tanımlanır ve uygulanır.

In [3]:
def clean_size(s):
    if isinstance(s, str):
        s = s.strip()
        if s.endswith('M'):
            try:
                return float(s[:-1])
            except:
                return np.nan
        if s.endswith('k'):
            try:
                return float(s[:-1]) / 1024.0  
            except:
                return np.nan
        if s == 'Varies with device':
            return np.nan
    return np.nan

def clean_installs(s):
    if isinstance(s, str):
        s = s.replace('+', '').replace(',', '')
        try:
            return int(s)
        except:
            return np.nan
    return np.nan

def clean_price(s):
    if isinstance(s, str):
        s = s.replace('$', '')
        try:
            return float(s)
        except:
            return np.nan
    return np.nan

In [4]:
df2 = df.copy()

df2['Size_Mb']        = df2['Size'].apply(clean_size)
df2['Installs_clean'] = df2['Installs'].apply(clean_installs)
df2['Price_clean']    = df2['Price'].apply(clean_price)


df2['Reviews'] = pd.to_numeric(df2['Reviews'], errors='coerce')


df2 = df2[~df2['Rating'].isna()].copy()


df2['High_Rating'] = (df2['Rating'] >= 4.0).astype(int)

3. Özellik ve Hedef Tanımlama
Modelde kullanılacak son özellikler (X) belirlenir ve hedef değişkenin (y) dağılımı incelenir.

In [5]:
features = [
    'Category', 'Reviews', 'Size_Mb', 'Installs_clean',
    'Type', 'Price_clean', 'Content Rating', 'Genres'
]

X = df2[features]
y = df2['High_Rating']

In [6]:
print("X shape:", X.shape)
print("Positive class ratio:", y.mean())

X shape: (9367, 8)
Positive class ratio: 0.7866979822782108


4. Veri Bölme ve Ön İşleme Hattı Oluşturma
Veri, eğitim ve test kümelerine ayrılır ve tüm ön işleme adımlarını (eksik doldurma, ölçekleme, kodlama) otomatik olarak uygulayacak bir ColumnTransformer oluşturulur.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (7493, 8) Test shape: (1874, 8)


In [8]:
numeric_features = ['Reviews', 'Size_Mb', 'Installs_clean', 'Price_clean']
categorical_features = ['Category', 'Type', 'Content Rating', 'Genres']

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


5. Özellik Seçimi (RFE) ve Model Pipeline'ları
Bu kısımda, model performansı için önemli 30 özelliği seçmek üzere RFE (Recursive Feature Elimination) tekniği hazırlanır ve bu özellik seçimi adımını da içeren üç farklı model pipeline'ı oluşturulur.

In [9]:
rfe_estimator = LogisticRegression(
    solver='liblinear',
    random_state=42,
    max_iter=1000
)

rfe = RFE(
    estimator=rfe_estimator,
    n_features_to_select=30, 
    step=1
)

6. Model Değerlendirmesi ve Sonuçlar
run_model fonksiyonu tanımlanır ve tüm pipeline'lar çalıştırılarak test verisi üzerindeki performansları analiz edilir.

In [10]:
def run_model(name, pipe):
    print("=" * 80)
    print(f"Model: {name}")
    print("=" * 80)

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    print(classification_report(y_test, y_pred))
    if hasattr(pipe, "predict_proba"):
        y_proba = pipe.predict_proba(X_test)[:, 1]
        print("ROC AUC:", roc_auc_score(y_test, y_proba))

    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


In [11]:
pipe_sami_knn = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rfe", rfe),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

pipe_sami_linsvc = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rfe", rfe),
    ("model", LinearSVC())
])

pipe_sami_gb = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rfe", rfe),
    ("model", GradientBoostingClassifier(random_state=42))
])


In [12]:
run_model("Sami - KNN + RFE", pipe_sami_knn)
run_model("Sami - LinearSVC + RFE", pipe_sami_linsvc)
run_model("Sami - GradientBoosting + RFE", pipe_sami_gb)

Model: Sami - KNN + RFE
              precision    recall  f1-score   support

           0       0.31      0.20      0.24       400
           1       0.80      0.88      0.84      1474

    accuracy                           0.74      1874
   macro avg       0.56      0.54      0.54      1874
weighted avg       0.70      0.74      0.71      1874

ROC AUC: 0.6429698100407055
Confusion Matrix:
 [[  79  321]
 [ 175 1299]]
Model: Sami - LinearSVC + RFE


E:\downloads\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
E:\downloads\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
E:\downloads\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.00      0.00      0.00       400
           1       0.79      1.00      0.88      1474

    accuracy                           0.79      1874
   macro avg       0.39      0.50      0.44      1874
weighted avg       0.62      0.79      0.69      1874

Confusion Matrix:
 [[   0  400]
 [   0 1474]]
Model: Sami - GradientBoosting + RFE
              precision    recall  f1-score   support

           0       0.67      0.01      0.01       400
           1       0.79      1.00      0.88      1474

    accuracy                           0.79      1874
   macro avg       0.73      0.50      0.45      1874
weighted avg       0.76      0.79      0.69      1874

ROC AUC: 0.7164196065128902
Confusion Matrix:
 [[   2  398]
 [   1 1473]]
